# RAM DCA — Recherche & Walk-Forward

Stratégie mean reversion multi-niveaux avec DCA :
- Plusieurs bandes enveloppe autour d'une MA
- Entrée progressive (plus le prix s'éloigne, plus on renforce)
- Sortie au retour à la MA
- Stop loss + cooldown post-SL

**Workflow :**
1. Backtest simple sur BTC pour vérifier la mécanique
2. Walk-forward + Optuna pour trouver des paramètres robustes
3. Analyse de la stabilité cross-fenêtres
4. Backtest final avec les meilleurs paramètres

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from vectorbtpro import vbt
from src.strategies.ram_dca import run_backtest
from config import MIN_TRADES, MAX_DRAWDOWN

## 1. Chargement des données

In [ ]:
PAIR     = 'BTCUSDT'
EXCHANGE = 'binance'
TF       = '1h'         # timeframe cible
TF_MIN   = 60           # en minutes

raw = pd.read_csv(f'../../data/raw/{EXCHANGE}/um/1m/{PAIR}.csv', low_memory=False)
raw['date'] = pd.to_datetime(raw['date'], unit='ms')
raw = raw.set_index('date').sort_index()
raw = raw[~raw.index.duplicated(keep='first')]

# Resample au timeframe cible
data = raw.resample(f'{TF_MIN}min').agg({
    'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum'
}).dropna(subset=['close'])

print(f'{PAIR} {TF} : {len(data):,} bougies  [{data.index[0].date()} → {data.index[-1].date()}]')
data.tail(3)

## 2. Backtest simple — vérification de la mécanique

On teste une config à 3 niveaux pour s'assurer que les signaux et la comptabilité sont corrects.

In [ ]:
# Config de référence : 3 niveaux, allocations croissantes
ENV_LEVELS  = [0.01, 0.02, 0.035]   # 1%, 2%, 3.5% de la MA
ALLOCATIONS = [0.20, 0.30, 0.50]    # 20% / 30% / 50% du capital
MA_WINDOW   = 100
SL_PCT      = 0.06

pf = run_backtest(data, MA_WINDOW, ENV_LEVELS, ALLOCATIONS, SL_PCT)
print(pf.stats())

In [ ]:
# Visualisation : prix + bandes + trades
import plotly.io as pio
pio.renderers.default = 'png'

from vectorbtpro import vbt as _vbt

ma_plot = _vbt.MA.run(data['close'], window=MA_WINDOW).ma.shift(1)

fig = pf.plot()
fig.update_traces(selector=dict(name='Close'), visible=False)

# Bougies
fig.add_trace(go.Candlestick(
    x=data.index, open=data['open'], high=data['high'],
    low=data['low'], close=data['close'], name='OHLC',
    increasing_line_color='#26a69a', decreasing_line_color='#ef5350',
))

# MA
fig.add_trace(go.Scatter(x=data.index, y=ma_plot, name='MA', line=dict(color='orange', width=1)))

# Bandes
colors = ['#4caf50', '#2196f3', '#9c27b0']
for i, (pct, col) in enumerate(zip(ENV_LEVELS, colors)):
    fig.add_trace(go.Scatter(
        x=data.index, y=ma_plot * (1 + pct),
        name=f'Upper {pct*100:.1f}%', line=dict(color=col, dash='dash', width=1), opacity=0.5
    ))
    fig.add_trace(go.Scatter(
        x=data.index, y=ma_plot * (1 - pct),
        name=f'Lower {pct*100:.1f}%', line=dict(color=col, dash='dash', width=1), opacity=0.5
    ))

fig.update_layout(xaxis_rangeslider_visible=False, template='plotly_dark', height=700)
fig.show()

## 3. Walk-Forward + Optuna

On optimise les paramètres sur 10 fenêtres glissantes (70% train / 30% test).

**Espace de recherche :**
- `ma_window` : période de la MA
- `env1, env2, env3` : distances des 3 bandes (en %)
- `sl_pct` : stop loss

Les allocations sont fixées à [0.20, 0.30, 0.50] pour réduire la dimensionnalité.

In [ ]:
def get_wf_windows(data, n_windows=10, split=0.7):
    """Découpe les données en n_windows fenêtres walk-forward sans chevauchement des tests."""
    n = len(data)
    window_size = n // n_windows
    train_size  = int(window_size * split)
    test_size   = window_size - train_size

    windows = []
    for i in range(n_windows):
        start      = i * window_size
        train_end  = start + train_size
        test_end   = train_end + test_size
        if test_end > n:
            break
        windows.append((data.iloc[start:train_end], data.iloc[train_end:test_end]))
    return windows

windows = get_wf_windows(data)
print(f'{len(windows)} fenêtres')
for i, (tr, te) in enumerate(windows):
    print(f'  F{i+1}: train [{tr.index[0].date()} → {tr.index[-1].date()}]  '
          f'test [{te.index[0].date()} → {te.index[-1].date()}]')

In [ ]:
FIXED_ALLOCS = [0.20, 0.30, 0.50]   # allocations fixes pour simplifier l'espace
N_TRIALS     = 200

results       = []
all_trials    = []

for i, (train, test) in enumerate(windows):
    print(f'\n--- Fenêtre {i+1}/{len(windows)} ---')

    def objective(trial):
        ma_w = trial.suggest_int('ma_window', 20, 300, step=10)
        e1   = trial.suggest_float('env1', 0.005, 0.02,  step=0.0025)  # 0.5% à 2%
        e2   = trial.suggest_float('env2', 0.015, 0.04,  step=0.0025)  # 1.5% à 4%
        e3   = trial.suggest_float('env3', 0.03,  0.08,  step=0.005)   # 3% à 8%
        sl   = trial.suggest_float('sl_pct', 0.03, 0.12, step=0.01)

        # Contrainte : niveaux strictement croissants
        if not (e1 < e2 < e3):
            return -999.0

        try:
            pf = run_backtest(train, ma_w, [e1, e2, e3], FIXED_ALLOCS, sl)
        except Exception:
            return -999.0

        if pf.trades.count() < MIN_TRADES:
            return -999.0
        if pf.max_drawdown > MAX_DRAWDOWN:
            return -999.0

        return float(pf.sharpe_ratio * 0.7 + pf.total_return * 0.3 * (1 - pf.max_drawdown))

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    bp = study.best_params
    print(f'  Meilleurs params: ma={bp["ma_window"]} '
          f'env=[{bp["env1"]*100:.2f}%, {bp["env2"]*100:.2f}%, {bp["env3"]*100:.2f}%] '
          f'sl={bp["sl_pct"]*100:.1f}%')

    for t in study.trials:
        if t.value is None: continue
        all_trials.append({'window': i+1, 'score': t.value, **t.params})

    pf_test = run_backtest(test, bp['ma_window'],
                           [bp['env1'], bp['env2'], bp['env3']],
                           FIXED_ALLOCS, bp['sl_pct'])
    results.append({
        'window':       i + 1,
        'ma_window':    bp['ma_window'],
        'env1':         bp['env1'], 'env2': bp['env2'], 'env3': bp['env3'],
        'sl_pct':       bp['sl_pct'],
        'train_score':  study.best_value,
        'test_sharpe':  pf_test.sharpe_ratio,
        'test_return':  pf_test.total_return,
        'test_max_dd':  pf_test.max_drawdown,
        'test_trades':  pf_test.trades.count(),
    })

df_results = pd.DataFrame(results)
df_trials  = pd.DataFrame(all_trials)
print('\nRésultats test :')
print(df_results[['window','ma_window','train_score','test_sharpe','test_return','test_max_dd','test_trades']])

## 4. Analyse de stabilité cross-fenêtres

On cherche les zones de l'espace des paramètres qui performent bien sur **plusieurs fenêtres simultanément** (robustesse).

In [ ]:
valid = df_trials[df_trials['score'] > -999].copy()

# Normalisation des paramètres pour la distance euclidienne
params_to_norm = ['ma_window', 'env1', 'env2', 'env3', 'sl_pct']
for col in params_to_norm:
    mn, mx = valid[col].min(), valid[col].max()
    valid[f'{col}_n'] = (valid[col] - mn) / (mx - mn) if mx > mn else 0.0

norm_cols = [f'{c}_n' for c in params_to_norm]
n_windows = df_results['window'].max()

# Top 30% par fenêtre
top_per_window = {}
for w in range(1, n_windows + 1):
    wt  = valid[valid['window'] == w]
    thr = wt['score'].quantile(0.70)
    top_per_window[w] = wt[wt['score'] >= thr].copy()

# Filtrage : garder uniquement les points présents dans 50%+ des fenêtres
SEUIL        = 0.35   # distance euclidienne normalisée
RATIO_MIN    = 0.50
RATIO_GREEN  = 0.70

points_stables = []
for w, top in top_per_window.items():
    for _, row in top.iterrows():
        voisins = 0
        for w2, top2 in top_per_window.items():
            if w2 == w: continue
            dists = np.sqrt(sum((top2[c] - row[c])**2 for c in norm_cols))
            if (dists <= SEUIL).any():
                voisins += 1
        ratio = voisins / (n_windows - 1)
        if ratio >= RATIO_MIN:
            points_stables.append({
                **{c: row[c] for c in params_to_norm},
                'score':   row['score'],
                'window':  w,
                'voisins': voisins,
                'ratio':   ratio,
                'color':   'green' if ratio >= RATIO_GREEN else 'orange',
            })

df_stable = pd.DataFrame(points_stables)
print(f'{len(df_stable)} points stables ({len(df_stable[df_stable["color"]=="green"])} verts, '
      f'{len(df_stable[df_stable["color"]=="orange"])} oranges)')

In [ ]:
if len(df_stable) == 0:
    print('Aucun point stable — paramètres trop instables entre fenêtres.')
else:
    fig = go.Figure()
    for color, label in [('orange', '50-70% fenêtres'), ('green', '>70% fenêtres')]:
        sub = df_stable[df_stable['color'] == color]
        if len(sub) == 0: continue
        fig.add_trace(go.Scatter(
            x=sub['ma_window'], y=sub['env1'],
            mode='markers',
            marker=dict(color=color, size=9, opacity=0.6),
            name=label,
            text=[f'score={s:.2f} env2={e2:.3f} env3={e3:.3f} sl={sl:.2f} voisins={v}/{n_windows-1}'
                  for s, e2, e3, sl, v in zip(sub['score'], sub['env2'], sub['env3'], sub['sl_pct'], sub['voisins'])]
        ))
    fig.update_layout(
        title='Zones robustes cross-fenêtres (ma_window vs env1)',
        xaxis_title='ma_window', yaxis_title='env1 (%)',
        template='plotly_dark', height=500
    )
    fig.show()

    # Paramètres finaux = médiane du meilleur cluster
    best_color = 'green' if len(df_stable[df_stable['color'] == 'green']) > 0 else 'orange'
    cluster    = df_stable[df_stable['color'] == best_color]
    final_params = {c: cluster[c].median() for c in params_to_norm}
    final_params['ma_window'] = int(round(final_params['ma_window'] / 10) * 10)
    print('\nParamètres finaux (médiane cluster) :')
    for k, v in final_params.items():
        print(f'  {k} = {v:.4f}')

## 5. Backtest final

On valide les paramètres sur **tout le dataset** (in-sample — conserver à l'esprit).

In [ ]:
pf_final = run_backtest(
    data,
    ma_window=final_params['ma_window'],
    envelope_levels=[final_params['env1'], final_params['env2'], final_params['env3']],
    allocations=FIXED_ALLOCS,
    sl_pct=final_params['sl_pct'],
)
print(pf_final.stats())

In [ ]:
# Plot final avec bandes
ma_final = vbt.MA.run(data['close'], window=final_params['ma_window']).ma.shift(1)

fig = pf_final.plot()
fig.update_traces(selector=dict(name='Close'), visible=False)
fig.add_trace(go.Candlestick(
    x=data.index, open=data['open'], high=data['high'],
    low=data['low'], close=data['close'], name='OHLC',
    increasing_line_color='#26a69a', decreasing_line_color='#ef5350',
))
fig.add_trace(go.Scatter(x=data.index, y=ma_final, name='MA', line=dict(color='orange', width=1)))

colors = ['#4caf50', '#2196f3', '#9c27b0']
for pct, col in zip([final_params['env1'], final_params['env2'], final_params['env3']], colors):
    fig.add_trace(go.Scatter(x=data.index, y=ma_final*(1+pct),
        name=f'+{pct*100:.2f}%', line=dict(color=col, dash='dash', width=1), opacity=0.5))
    fig.add_trace(go.Scatter(x=data.index, y=ma_final*(1-pct),
        name=f'-{pct*100:.2f}%', line=dict(color=col, dash='dash', width=1), opacity=0.5))

fig.update_layout(xaxis_rangeslider_visible=False, template='plotly_dark', height=750)
fig.show()

## 6. Corrélation train/test — check overfitting

In [ ]:
corr = df_results['train_score'].corr(df_results['test_sharpe'])
print(f'Corrélation train score / test sharpe : {corr:.3f}')

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_results['train_score'], y=df_results['test_sharpe'],
    mode='markers+text',
    text=[f'F{w}' for w in df_results['window']],
    textposition='top center',
    marker=dict(size=10, color=df_results['test_sharpe'],
                colorscale='RdYlGn', showscale=True),
))
fig.add_hline(y=0, line_dash='dash', line_color='white', opacity=0.4)
fig.update_layout(
    title=f'Train score vs Test sharpe (corrélation = {corr:.3f})',
    xaxis_title='Train score', yaxis_title='Test Sharpe',
    template='plotly_dark', height=450
)
fig.show()
print(df_results[['window','test_sharpe','test_return','test_max_dd','test_trades']])